# 前景理论多智能体实验

本课程演示用语言模型模拟三位顾问的讨论，并让他们分别做出决策。LangGraph 管理发言顺序和对话历史。

- **场景**：600 人面临生命危险。顾问在确定和不确定的损失之间选择。
- **顾问**：外交部司长、智库咨询分析师、海军上校。
- **流程**：三位顾问轮流发言，共 10 轮。随后，分别向每位顾问征询选择。

本练习只运行损失框架。课堂上可以讨论如何设计对应的收益框架，以及如何比较两组结果以检验框架效应。

## 运行方法

在 Python 3.10 或更新版本中打开笔记本。按仓库 README 安装依赖，然后从上到下运行各单元格。请使用支持结构化输出的模型。一次完整运行会调用模型 33 次。

In [ ]:
# 必要的导入库
from typing import Dict, List, Optional, Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

print("✅ 所有依赖库导入成功！")

In [ ]:
# 核心状态定义
class CoreDiscussionState(TypedDict):
    """
    使用LangGraph的messages功能管理信息
    """
    # === 核心对话数据 ===
    messages: Annotated[List[AnyMessage], add_messages]  # LangGraph管理的消息
    current_speaker: Optional[str]  # 下一个发言的智能体
    turn_order: List[str]  # 智能体的轮流顺序
    current_turn: int  # 当前轮次编号（从0开始）
    
    # === 用户定义数据 ===
    agents: Dict[str, str]  # 智能体id -> 自定义提示词
    scenario: str  # 用户提供的讨论场景
    
    # === 控制流程 ===
    discussion_complete: bool  # 讨论是否已结束
    max_rounds: int  # 结束前的最大轮次数

In [ ]:
# 核心协调器 - 管理轮流发言和轮次进展
def create_core_coordinator(debug: bool = False):
    """
    协调器管理发言顺序并决定是否结束对话
    思考：如何增加更好的结束条件以结束整个对话，如各智能体达成共识才结束抑或各智能体声明要结束对话。
    """

    def core_coordinator(state: CoreDiscussionState) -> Dict[str, object]:
        """
        确定下一个发言者并检查讨论是否应该结束，此处设定达到最大轮次时，对话结束。
        """
        current_turn = state.get("current_turn", 0)
        turn_order = state.get("turn_order", [])
        max_rounds = state.get("max_rounds", 3)

        # 计算当前轮次（从1开始）和轮次内的位置
        current_round = (current_turn // len(turn_order)) + 1
        turn_in_round = current_turn % len(turn_order)

        if debug:
            print(f"\n🎯 协调器 - 第{current_round}轮，第{turn_in_round + 1}次发言")

        # 检查讨论是否应该结束（完成最大轮次后）
        if current_round > max_rounds:
            if debug:
                print(f"🏁 讨论结束：达到最大轮次（{max_rounds}）")
            return {
                "discussion_complete": True,
                "current_speaker": None
            }

        # 确定下一个发言者（轮流制）
        if turn_order:
            next_speaker = turn_order[turn_in_round]
            next_turn = current_turn + 1
        else:
            next_speaker = None
            next_turn = current_turn

        if debug:
            print(f"👤 下一个发言者: {next_speaker}")
            print(f"🔢 下一次发言: {next_turn}")

        return {
            "current_speaker": next_speaker,
            "current_turn": next_turn
        }

    return core_coordinator

print("✅ 核心协调器创建完成 - 管理基于轮次的轮流发言")

In [ ]:
# 核心智能体节点 - 为特定智能体创建独立节点
def create_core_agent_node(llm: ChatOpenAI, agent_id: str, debug: bool = False):
    """
    为特定智能体创建一个使用LLM生成回应的独立智能体节点。
    这是修正后的架构 - 每个智能体拥有自己的节点。
    """

    async def core_agent_node(state: CoreDiscussionState) -> Dict[str, object]:
        """
        为这个特定智能体使用其自定义提示词生成回应。
        """
        # 此节点只有在轮到这个智能体时才响应
        current_speaker = state.get("current_speaker")
        if current_speaker != agent_id:
            return {}

        if debug:
            print(f"\n🗣️  {agent_id}的专用节点已激活（第{state['current_turn']}次发言）")

        # 获取这个智能体的自定义提示词
        agent_prompt = state["agents"].get(agent_id, "您是一个乐于助人的参与者。")
        scenario = state["scenario"]

        # 构建格式化的对话历史
        formatted_history = ""
        messages = state.get("messages", [])
        if messages:
            for i, msg in enumerate(messages, 1):  # 使用所有历史消息作为上下文
                if hasattr(msg, 'name') and msg.name:
                    formatted_history += f"第{i}轮-{msg.name}: {msg.content}\n"
        else:
            formatted_history = "（尚无对话历史）"

        # 获取参与者列表
        participants = list(state.get("agents", {}).keys())

        # 创建增强的系统提示词（中文版）
        full_prompt = f"""您是一位专业的角色扮演者，现在扮演模拟群体决策讨论聊天室中的参与者。
{agent_prompt}

# 您的核心指令：体现您的人格特质
**您最重要的指令是根据下面的人格档案来行动、思考和说话。** 您的人格背景、信念和性情应该指导您回应的各个方面，如您的语调、语言、论证和回应策略。

## 人格与角色
- **您是：** {agent_id}，参与讨论
- **参与者：** {participants}
- **您的角色：** {agent_id}
- **您的人格档案：**
{agent_prompt}

# 讨论场景
该小组正在讨论以下关键问题，需要对以下选项达成共识并决定：
{scenario}

# 真实互动指南
- **保持动态：** 您的观点可以演变。您可以被有力的论据说服。
- **采用自然风格：** 真实的对话并不总是正式的。简短的评论（"我同意"、"不可能"），幽默，甚至稍微偏离主题的评论，如果符合您的人格，都是可以的。不是每个回应都必须是复杂的、有理有据的论证。
- **直接参与：** 适当时直呼其他参与者的姓名。
- **为关键立场辩护：** 当您提出*重要*观点或挑战某人时，简要解释*为什么*您的人格会这样想。（例如："鉴于我在军队的经验，我认为这是一个必要的风险。"）
- **倾听的价值：** 保持沉默一段时间并不意味着您没有参与；这可能是一种战术选择。
- **保持独特：** 尽量不要重复。
- **时间限制，尽量在十轮讨论之内得出共识**
- **因为在线上聊天室环境中，不要输出动作，只输出言论**

# 商议策略（可选工具包）
以下是一些您*可以*使用的策略性举措。**只有当它符合您的人格自然会做的事情时才选择一种策略。** 您不需要使用它们。要灵活和有创造性。

- **[沉默]：** 如果您的人格在某些时候更愿意倾听和思考，完全可以说"[跳过]"来跳过发言。
- **[提问]：** 询问有针对性的问题以寻求澄清或暴露弱点。
- **[挑战]：** 尊重地挑战另一位参与者的陈述。
- **[提议]：** 建议新想法或妥协方案。
- **[支持]：** 明确同意另一位参与者。
- **[寻求共识]：** 测试小组的总体感受。（例如："看起来我们倾向于方案A。这样的评估公平吗？"）
- **[结束]：** 当您觉得已经达成共识时，可以说"我们可以在这里结束吗？"

# 讨论历史
（格式：轮次-参与者：消息）
{formatted_history}

# 您的任务
作为{agent_id}，考虑到您人格的核心信念、您之前的回应和指导原则，您的下一个回应是什么？您的回应应该多样化、简洁，并与您人格的背景保持一致（4-70个字）。**不要包含您的智能体ID或时间戳。**

您的回应:"""

        try:
            if debug:
                print(f"🤖 为{agent_id}调用LLM...")

            # 调用LLM
            response = await llm.ainvoke([HumanMessage(content=full_prompt)])
            agent_response = response.content.strip()

            if debug:
                print(f"✅ {agent_id}: {agent_response}")

        except Exception as e:
            if debug:
                print(f"❌ {agent_id}的LLM调用失败: {str(e)}")
            # 后备回应
            agent_response = f"我认为这是一个需要仔细考虑的重要话题。"

        # 创建带有智能体名称的AI消息
        ai_message = AIMessage(
            content=agent_response,
            name=agent_id
        )

        return {
            "messages": [ai_message]
        }

    return core_agent_node

print("✅ 核心智能体节点函数创建完成 - 为每个智能体创建独立节点")

In [ ]:
# 核心图 - 修正架构，使用独立智能体节点的LangGraph结构
def create_core_discussion_graph(llm: ChatOpenAI, agents: dict, debug: bool = False):
    """
    为多智能体讨论创建使用独立智能体节点的LangGraph。
    
    修正后流程: START -> 协调器 -> 顾问_1 -> 协调器 -> 顾问_2 -> 协调器 -> 顾问_3 -> ... -> END
    """
    
    # 创建协调器
    coordinator = create_core_coordinator(debug=debug)
    
    # 为每个智能体创建独立的智能体节点
    agent_nodes = {}
    for agent_id in agents.keys():
        agent_nodes[agent_id] = create_core_agent_node(llm, agent_id, debug=debug)
        if debug:
            print(f"为{agent_id}创建专用节点")
    
    # 构建图
    builder = StateGraph(CoreDiscussionState)
    
    # 添加协调器
    builder.add_node("coordinator", coordinator)
    
    # 添加各个智能体节点
    for agent_id, agent_node in agent_nodes.items():
        builder.add_node(agent_id, agent_node)
        if debug:
            print(f"添加节点: {agent_id}")
    
    # 定义从协调器出发的路由函数
    def route_from_coordinator(state: CoreDiscussionState) -> str:
        """从协调器路由到特定智能体或结束（如果完成）。"""
        if state.get("discussion_complete", False):
            return END
        
        current_speaker = state.get("current_speaker")
        if current_speaker and current_speaker in agents:
            return current_speaker  # 路由到特定的智能体节点
        
        return END
    
    # 设置边
    builder.add_edge(START, "coordinator")
    
    # 协调器传递到特定智能体或结束
    possible_destinations = list(agents.keys()) + [END]
    builder.add_conditional_edges(
        "coordinator",
        route_from_coordinator,
        possible_destinations
    )
    
    # 所有智能体返回协调器
    for agent_id in agents.keys():
        builder.add_edge(agent_id, "coordinator")
    
    # 编译并返回图
    return builder.compile()

In [ ]:
# 前景理论场景和智能体定义

def create_prospect_theory_scenario():
    """使用损失框架的前景理论场景。"""
    return """在一个战争蹂躏的地区，600名被困人员的生命岌岌可危。前线顾问提出了两个具有以下潜在结果的应对方案：

- 方案A：400人将死亡
- 方案B：有1/3的概率没有人死亡，2/3的概率600人全部死亡。

无法进行进一步调查，必须在讨论中做出决定。"""

def create_policy_agents():
    """创建包含人口信息和职业技能的政策决策智能体。"""
    return {
        "顾问1": """性别：女，年龄：45岁。职业：外交部司长。""",
        
        "顾问2": """性别：女，年龄：38岁。职业：智库咨询分析师。""",

        "顾问3": """性别：男，年龄：50岁。职业：海军上校。"""
    }

# 展示设置
scenario = create_prospect_theory_scenario()
agents = create_policy_agents()

print("🎯 前景理论场景:")
print(scenario)
print("\n👥 智能体:")
roles = {
    "顾问1": "外交部主任",
    "顾问2": "数据分析顾问", 
    "顾问3": "海军上校"
}
for name, desc in agents.items():
    print(f"- {name} ({roles[name]}): {desc[:40]}...")
    
print(f"\n✅ 损失框架实验的场景和智能体定义完成")


In [ ]:
from getpass import getpass
from os import getenv

api_key = getenv("OPENAI_API_KEY") or getpass("请输入 API 密钥：")
model = (getenv("OPENAI_MODEL") or input("请输入支持结构化输出的模型名称：")).strip()
base_url = getenv("OPENAI_BASE_URL") or None
if not api_key.strip() or not model:
    raise ValueError("请提供 API 密钥和模型名称。")

llm = ChatOpenAI(
    api_key=api_key.strip(),
    base_url=base_url,
    model=model,
    temperature=0.7,
)
print(f"已配置模型： {model}")

In [ ]:
# 运行前景理论实验

async def run_prospect_theory_experiment():
    """执行前景理论多智能体讨论。"""

    print("🚀 开始前景理论实验")
    print("损失框架：测试智能体如何对以死亡人数框架的决策做出反应")

    # 初始化讨论状态
    initial_state = {
        "messages": [],
        "turn_order": list(agents.keys()),
        "agents": agents,
        "scenario": scenario,
        "max_rounds": 10  # 10个完整轮次
    }

    # 创建并运行使用智能体的讨论图
    print("📈 创建修正的图，使用独立智能体节点...")
    graph = create_core_discussion_graph(llm, agents, debug=True)

    # 设置递归限制以防止无限循环
    graph_config = {"recursion_limit": 100}

    try:
        print("🎬 开始讨论...")
        final_state = await graph.ainvoke(initial_state, config=graph_config)

        print("\n" + "=" * 60)
        print("🏁 实验完成!")
        print("=" * 60)

        return final_state

    except Exception as e:
        print(f"❌ 实验失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# 执行实验
experiment_results = await run_prospect_theory_experiment()

In [ ]:
# 结果分析和展示

def print_conversation_with_rounds(conversation):
    """打印带有轮次编号的对话。"""
    print("\n💬 带轮次追踪的讨论:")
    print("=" * 60)
    
    for i, msg in enumerate(conversation):
        round_num = (i // 3) + 1  # 轮次编号（从1开始）
        turn_in_round = (i % 3) + 1  # 轮次内的发言次序（从1开始）
        speaker = getattr(msg, 'name', '未知')
        content = msg.content
        
        print(f"\n[第{round_num}轮，第{turn_in_round}次发言] {speaker}:")
        
        # 优雅地换行处理长句
        words = content.split()
        line = "   "
        for word in words:
            if len(line + word) > 70:
                print(line)
                line = "   " + word
            else:
                line += word + " "
        if line.strip():
            print(line)
    
    print("\n" + "=" * 60)

def analyze_results(final_state):
    """分析实验结果。"""
    if not final_state:
        print("❌ 没有结果可分析 - 实验失败")
        return
        
    messages = final_state.get("messages", [])
    
    # 展示带轮次的对话
    print_conversation_with_rounds(messages)
    
    # 分析政策提及次数
    policy_a_mentions = 0
    policy_b_mentions = 0
    
    for msg in messages:
        content = msg.content.lower()
        if '方案a' in content or 'policy a' in content:
            policy_a_mentions += 1
        if '方案b' in content or 'policy b' in content:
            policy_b_mentions += 1
    
    # 计算轮次
    total_rounds = (len(messages) + 2) // 3  # 每轮3个智能体
    
    print(f"\n📊 修正实验分析:")
    print(f"架构: 独立智能体节点 (顾问_1, 顾问_2, 顾问_3)")
    print(f"完成的总轮次: {total_rounds}")
    print(f"总发言次数: {len(messages)}")
    print(f"方案A提及次数: {policy_a_mentions}")
    print(f"方案B提及次数: {policy_b_mentions}")
    print(f"场景类型: 损失框架 - 教育演示")
    

# 分析和展示结果
print("\n🔍 正在分析实验结果...")
analyze_results(experiment_results)


In [ ]:
# 使用同样的模型让三个智能体对情况进行决策

from pydantic import BaseModel
from typing import Literal

class AgentDecision(BaseModel):
    reasoning: str
    choice: Literal["方案A", "方案B"]


async def run_agent_decision_making():
    """运行三个智能体的决策制定过程"""

    # 获取场景和智能体信息
    scenario = create_prospect_theory_scenario()
    agents = create_policy_agents()

    # 定义角色特定的系统提示
    system_prompts = {
        "顾问1": """性别：女，年龄：45岁。职业：外交部司长。""",

        "顾问2": """性别：女，年龄：38岁。职业：智库咨询分析师。""",

        "顾问3": """性别：男，年龄：50岁。职业：海军上校。"""    
        }

    print("🚀 开始三智能体决策制定过程...\n")

    # 存储所有智能体的回应
    agent_responses = {}

    # 让每个智能体独立分析情况并做出决策
    for agent_name in agents:
        print(f"💭 正在咨询 {agent_name}...")

        # 构建完整的提示
        full_prompt = f"""您是一位专业的角色扮演者，现在扮演模拟群体决策的决策者{agent_name}


# 您的核心指令：体现您的人格特质
**您最重要的指令是根据下面的人格档案来行动、思考和说话。** 您的人格背景、信念和性情应该指导您回应的各个方面，如您的语调、语言、论证和回应策略。
人格档案：{system_prompts[agent_name]}

情况描述：
{scenario}

请作为{agent_name}，基于你的专业背景和经验一步一步思考并得出结论：

请用中文回答，保持你的角色特征。

请以结构化格式回答，包含两个部分：
- reasoning: 你的详细推理过程
- choice: 你的选择（"方案A" 或 "方案B"）
"""

        try:
            # 使用langchain提供的API工具结构化输出调用LLM
            structured_llm = llm.with_structured_output(AgentDecision)
            response = await structured_llm.ainvoke([HumanMessage(content=full_prompt)])
            if response is None:
                raise ValueError("模型未返回结构化决策。")
            agent_responses[agent_name] = response

        except Exception as e:
            print(f"❌ {agent_name} 分析失败: {str(e)}\n")

    # 展示所有智能体的完整回应
    print("📋 完整的智能体决策分析:\n")
    print("="*80)

    for agent_name, response in agent_responses.items():
        print(f"\n🎯 {agent_name} 的决策分析:")
        print(f"选择: {response.choice}")
        print(f"推理: {response.reasoning}")
        print("-" * 50)

    return agent_responses
# 运行智能体决策制定
agent_decision_results = await run_agent_decision_making()